# τ → 3μ transformer: training on SWAN

Start SWAN with a **GPU** software stack (an LCG release with CUDA). The training set made by
`scripts/make_training_set.py` must be on EOS (see `README.md`). Training writes a checkpoint after
every epoch: if the session drops, rerun the training cell and it resumes.


In [ ]:
import os, sys, json
REPO = os.path.expanduser("~/Tau3MuRun3_NANO")      # git clone -b transformer-ntuples https://github.com/caruta/Tau3MuRun3_NANO
DATA = "/eos/user/X/XXXX/tau3mu/train_Transformer_v1"  # training set copied from Bari
OUT  = "/eos/user/X/XXXX/tau3mu/runs/v1"               # checkpoints, metrics and plots
sys.path.insert(0, os.path.join(REPO, "PhysicsTools/Tau3muML"))

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU!)")
stats = json.load(open(os.path.join(DATA, "stats.json")))
print("shards:", stats["n_shards"], "| rows:", {k: v["total"] for k, v in stats["counts"].items()})


## Configuration

`disco` is the weight of the mass-decorrelation term: raise it if the sculpting checks below fail,
lower it if the separation suffers a lot. `prod_frac` sets the signal production mix
(1 Dₛ, 2 B→Dₛ, 3 B→τ, 4 D⁺, 5 W); `None` keeps the MC mix. For a first look, `max_train_rows` limits
each epoch.


In [ ]:
cfg = dict(epochs=20, batch_size=256, lr=5e-4,
           disco=2.0, disco_on="score",   # "logit": decorrelate the logit, which also sees the high-score tail
           drop=None,                      # inputs to ignore, e.g. ["trk:log_ptrel", "neu:log_ptrel", "sv:log_ptrel", "jet:log_ptrel"]
           drop_tokens=None,               # token types to leave out, e.g. ["trk", "neu", "sv", "jet"]
           d=128, heads=8, layers=6, dropout=0.1,
           prod_frac=None, npv_reweight=True,
           max_train_rows=None)
FOLDS = [0, 1, 2, 3, 4]


In [ ]:
from tau3muml.train import train
for k in FOLDS:
    train(DATA, OUT, k, **cfg)


## Training curves


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for k in FOLDS:
    p = os.path.join(OUT, f"fold{k}", "metrics.csv")
    if not os.path.exists(p):
        continue
    m = pd.read_csv(p)
    axs[0].plot(m.epoch, m.train_bce, "--", label=f"train {k}"); axs[0].plot(m.epoch, m.val_bce, label=f"val {k}")
    axs[1].plot(m.epoch, m.val_auc, label=f"fold {k}")
    axs[2].plot(m.epoch, m.val_disco, label=f"fold {k}")
for ax, t in zip(axs, ["BCE", "val AUC", "val DisCo (bkg score vs mass)"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=7)
plt.show()


## Out-of-fold evaluation and sculpting checks

* `bkg_mass.png`: sideband mass shape for different score ranges. The shapes should agree.
* `ss3mu_mass.png` and `ss3mu_SR_over_SB` in the summary: the |Q| = 3 candidates contain no signal and are
  not blinded. If the SR/SB ratio rises with tighter cuts (beyond the errors), the model has learnt the mass.
* `npv.png`: efficiency vs number of PVs. A strong slope in signal but not background points to a pileup
  shortcut (signal is MC, background is data).


In [ ]:
from tau3muml.evaluate import evaluate
summary, scores = evaluate(DATA, OUT)


In [ ]:
from IPython.display import Image, display
for p in ["roc.png", "score.png", "bkg_mass.png", "ss3mu_mass.png", "npv.png"]:
    display(Image(os.path.join(OUT, "eval", p)))


## What the model relies on

Permutation tests on a held-out sample of one fold: each input group is shuffled across candidates (or a
token type removed) and the change in AUC, background efficiency, sideband sculpting (`near_far`, 1 = none)
and data-taking dependence (`early_late`, 1 = none) is printed. About 30 s per group on a GPU.


In [ ]:
from tau3muml.diagnose import run as diagnose
rows = diagnose(DATA, OUT, fold=0, n_sig=30000, n_bkg=60000)


## Quick comparison with BDT baselines

BDTs (XGBoost, on the GPU if available) trained on the same candidates, folds and weights:
`bdt_run2` (Run 2 variables and settings; needs `bdt_run2.npz` from `scripts/make_bdt_table.py`, copied
into `DATA`), `bdt_run2like` and `bdt_candmu` (all candidate and muon features, no context tokens).
Output: sideband efficiency at fixed signal efficiency, SS3MU sculpting check, and a rough expected upper
limit on the signal yield as a ratio to `bdt_run2` (inclusive, and in the Run 2 categories).

`BKG_SCALE` = sideband candidates in the full data / in the training set, about 120 for this training set.
The ratios between methods depend only weakly on it.


In [ ]:
# if xgboost is missing: !pip install --user xgboost  (falls back to scikit-learn otherwise)
from tau3muml.compare_bdt import compare
BKG_SCALE = 120
res = compare(DATA, OUT, BKG_SCALE)
display(Image(os.path.join(OUT, "eval", "compare_roc.png")))
